# Real Estate Buyer Segmentation & Investment Profiling
### Unsupervised Machine Learning Market Intelligence for Parcl Co. Limited

This notebook contains the complete step-by-step exploratory analysis, feature engineering, model training, validation, and profiling workflow for Parcl Co. Limited.

## Step 0: Set Up Environment & Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project packages are accessible
sys.path.append(os.path.abspath('../src'))
import config
import preprocessing
import modeling
import visualization

sns.set_theme(style="whitegrid")
print("Setup Complete.")

## Step 1: Load and Clean Datasets

We merge the client demographic details and the property specifications. Then, we apply deduplication, impute missing entries, and extract customer Age from Date of Birth.

In [ ]:
# 1. Load Raw
df_merged = preprocessing.load_raw_data()
print("Merged Dataset Raw Shape:", df_merged.shape)

# 2. Apply Cleaning Pipeline
df_cleaned = preprocessing.clean_data(df_merged)
print("Cleaned Dataset Shape:", df_cleaned.shape)
df_cleaned.head()

## Step 2: Exploratory Data Analysis (EDA)

We visualize univariate countplots for demographics, distributions for finances (price/size), and plot numeric correlations.

In [ ]:
# Plot missing value structure
visualization.plot_missing_values_heatmap(df_merged, show=True)

# Plot age & price distributions
visualization.plot_univariate_distribution(df_cleaned, config.COL_AGE, "Buyer Age Distribution", is_numeric=True, show=True)
visualization.plot_univariate_distribution(df_cleaned, config.COL_PURCHASE_PRICE, "Property Purchase Price (USD) Distribution", is_numeric=True, show=True)

# Plot country of origins
visualization.plot_univariate_distribution(df_cleaned, config.COL_COUNTRY, "Client Origin Distribution (Country)", show=True)

# Plot numeric correlations
visualization.plot_correlation_heatmap(df_cleaned, show=True)

## Step 3: Feature Engineering

We standardize continuous values using StandardScaler, map `loan_applied` to binary indicators, and one-hot encode categorical features.

In [ ]:
X_processed, feature_names, scaler, encoder = preprocessing.fit_transform_features(df_cleaned, is_training=True)
print("Processed Feature matrix shape:", X_processed.shape)
print("Processed Features Sample list:", feature_names[:10])

## Step 4 & 5: Clustering Validation and Optimal K Selection

We run KMeans clustering across K values ranging from 2 to 8. We analyze metrics (Inertia, Silhouette score, Davies-Bouldin index, Calinski-Harabasz score) to locate the peak and select the best partition programmatically.

In [ ]:
metrics_df = modeling.run_clustering_metrics(X_processed, max_k=8)

# Plot optimal select curves
visualization.plot_elbow_curve(metrics_df, show=True)
visualization.plot_validation_metrics(metrics_df, show=True)

# Auto determine
optimal_k = modeling.determine_optimal_k(metrics_df)
print("Recommended Optimal Clusters K:", optimal_k)

## Step 6: Cluster Analysis & Profiling

We fit the final KMeans model with the optimal K. We group customer attributes by cluster, analyze the centroids (average features), and dynamically label them.

In [ ]:
kmeans_model, labels = modeling.fit_kmeans_model(X_processed, optimal_k)

# Profiles identification
cluster_profiles = modeling.identify_cluster_profiles(df_cleaned, labels)

# Create final dataframe
df_labeled = df_cleaned.copy()
df_labeled["cluster_id"] = labels
df_labeled["cluster_name"] = df_labeled["cluster_id"].map(cluster_profiles)
df_labeled.head()

## Step 7: Dimensionality Reduction & Visualizations

We project dimensions using PCA to visualize the cohorts on 2D space, and render boxplots comparing customer metrics.

In [ ]:
pca, X_pca = modeling.apply_pca(X_processed, n_components=3)

# Visual PCA space
visualization.plot_pca_2d(X_pca, labels, cluster_profiles, show=True)

# Visual cohort size distributions
visualization.plot_cluster_sizes(df_labeled, show=True)

# Bivariate plots
visualization.plot_cluster_bivariate(df_labeled, config.COL_COUNTRY, title="Segment Origin Breakdown", show=True)
visualization.plot_cluster_bivariate(df_labeled, config.COL_LOAN_APPLIED, title="Segment Financed Loan Breakdown", show=True)

# Age boxplot
visualization.plot_cluster_boxplots(df_labeled, config.COL_AGE, show=True)

## Step 8: Save Pipeline Models

In [ ]:
modeling.save_ml_artifacts(kmeans_model, scaler, encoder)
print("All pipeline artifacts saved successfully in models/ directory.")